# Exploratory Data Analysis (EDA) on CHV PPE Dataset
This notebook performs exploratory data analysis on the CHV Personal Protective Equipment (PPE) dataset.

## 1. Import Libraries and Define Paths
We start by importing the necessary libraries and defining the paths to our dataset using `pathlib`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image, ImageDraw
import cv2
import random

# Define Paths
BASE_DIR = Path('../data/CHV_dataset')
IMG_DIR = BASE_DIR / 'images'
ANN_DIR = BASE_DIR / 'annotations'
SPLIT_DIR = BASE_DIR / 'data split'
REPORT_DIR = Path('../reports')

# Create reports directory if it doesn't exist
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Classes based on the README.md in annotations
CLASSES = {
    0: 'person',
    1: 'vest',
    2: 'blue helmet',
    3: 'red helmet',
    4: 'white helmet',
    5: 'yellow helmet'
}

## 2. Count Total Images and Annotation Files
Let's see how many images and annotation files we have in our dataset.

In [ ]:
image_files = [f for f in IMG_DIR.iterdir() if f.is_file() and f.suffix.lower() in ['.jpg', '.jpeg', '.png'] and not f.name.startswith('.DS_Store')]
annotation_files = [f for f in ANN_DIR.iterdir() if f.is_file() and f.suffix.lower() == '.txt' and f.name != 'README.md' and not f.name.startswith('.DS_Store')]

print(f"Total Images: {len(image_files)}")
print(f"Total Annotation Files: {len(annotation_files)}")

## 3. Check Image Extensions and Resolutions
We will sample a few images to check their extensions and resolutions to understand if they are uniform.

In [ ]:
from collections import Counter

extensions = [f.suffix.lower() for f in image_files]
print("Image Extensions Distribution:", Counter(extensions))

# Check resolutions for a subset of images (e.g., up to 100 images) to avoid long processing times
resolutions = []
for img_path in image_files[:100]:
    with Image.open(img_path) as img:
        resolutions.append(img.size) # (width, height)

print("Sample Image Resolutions (Top 5):", Counter(resolutions).most_common(5))

## 4. Inspect Annotation Format and Sample Content
The dataset uses the YOLO annotation format (`<class> <x_center> <y_center> <width> <height>`). We'll read and print a sample annotation.

In [ ]:
sample_ann = annotation_files[0] if annotation_files else None

if sample_ann:
    print(f"Content of {sample_ann.name}:")
    with open(sample_ann, 'r') as f:
        print(f.read())
else:
    print("No annotation files found.")

## 5. Extract Class Labels from Annotations
We'll parse all annotation files to gather data about every object bounded box to analyze class distributions and bounding box sizes.

In [ ]:
all_annotations = []

for ann_path in annotation_files:
    with open(ann_path, 'r') as f:
        lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                class_id = int(parts[0])
                x_center, y_center, width, height = map(float, parts[1:])
                all_annotations.append({
                    'image_id': ann_path.stem,
                    'class_id': class_id,
                    'class_name': CLASSES.get(class_id, f'Unknown ({class_id})'),
                    'x_center': x_center,
                    'y_center': y_center,
                    'width': width,
                    'height': height,
                    'area': width * height
                })

df_ann = pd.DataFrame(all_annotations)
print(f"Total Bounding Boxes: {len(df_ann)}")
df_ann.head()

## 6. Show Class Distribution Chart
Let's visualize how many instances we have for each PPE class. This helps identify class imbalances.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df_ann, y='class_name', order=df_ann['class_name'].value_counts().index, palette='viridis')
plt.title('Distribution of Classes in Dataset')
plt.xlabel('Count')
plt.ylabel('Class Name')
plt.tight_layout()

# Save chart to reports folder
plt.savefig(REPORT_DIR / 'class_distribution.png')
plt.show()

# Save summary to CSV
class_counts = df_ann['class_name'].value_counts().reset_index()
class_counts.columns = ['class_name', 'count']
class_counts.to_csv(REPORT_DIR / 'class_distribution.csv', index=False)

## 7. Check Missing Annotations for Images
We need to ensure every image has a corresponding annotation file, or at least identify images that don't.

In [ ]:
image_stems = set([f.stem for f in image_files])
annotation_stems = set([f.stem for f in annotation_files])

missing_annotations = image_stems - annotation_stems
print(f"Images without annotations: {len(missing_annotations)}")
if missing_annotations:
    print(f"Sample images missing annotations: {list(missing_annotations)[:5]}")

## 8. Check Annotations Without Matching Images
Conversely, we should check if there are any orphan annotation files that don't belong to any image.

In [ ]:
missing_images = annotation_stems - image_stems
print(f"Annotations without images: {len(missing_images)}")
if missing_images:
    print(f"Sample annotations missing images: {list(missing_images)[:5]}")

## 9. Analyze Bounding Box Width, Height, and Area Distribution
Let's see the distribution of bounding box sizes. This is crucial for configuring anchor boxes in object detection models like YOLO.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df_ann['width'], bins=50, ax=axes[0], color='skyblue')
axes[0].set_title('Bounding Box Width Distribution')

sns.histplot(df_ann['height'], bins=50, ax=axes[1], color='lightgreen')
axes[1].set_title('Bounding Box Height Distribution')

sns.histplot(df_ann['area'], bins=50, ax=axes[2], color='salmon')
axes[2].set_title('Bounding Box Area Distribution')

plt.tight_layout()
plt.savefig(REPORT_DIR / 'bbox_distribution.png')
plt.show()

## 10. Visualize Random Sample Images with Bounding Boxes
Let's randomly pick an image and draw the bounding boxes using the annotations. The YOLO format gives us center coordinates and relative sizes.

In [ ]:
def draw_boxes(image_path, annotations, class_mapping):
    with Image.open(image_path) as img:
        img_w, img_h = img.size
        draw = ImageDraw.Draw(img)
        
        for ann in annotations:
            cls_id, x_c, y_c, w, h = ann
            
            # Convert YOLO relative format to absolute pixel format
            abs_x_c = x_c * img_w
            abs_y_c = y_c * img_h
            abs_w = w * img_w
            abs_h = h * img_h
            
            x1 = abs_x_c - (abs_w / 2)
            y1 = abs_y_c - (abs_h / 2)
            x2 = abs_x_c + (abs_w / 2)
            y2 = abs_y_c + (abs_h / 2)
            
            draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
            draw.text((x1, y1 - 10), class_mapping.get(int(cls_id), str(cls_id)), fill="red")
            
        return img

# Pick a random image that has annotations
if image_stems.intersection(annotation_stems):
    sample_image = random.choice(list(image_stems.intersection(annotation_stems)))

    # Get annotations for this image
    sample_ann_path = ANN_DIR / f"{sample_image}.txt"
    sample_img_path = list(IMG_DIR.glob(f"{sample_image}.*"))[0]

    with open(sample_ann_path, 'r') as f:
        anns = [list(map(float, line.strip().split())) for line in f.readlines()]

    img_with_boxes = draw_boxes(sample_img_path, anns, CLASSES)
    plt.figure(figsize=(10, 10))
    plt.imshow(img_with_boxes)
    plt.axis('off')
    plt.title(f"Sample Annotations for {sample_image}")
    plt.savefig(REPORT_DIR / 'sample_visualization.png')
    plt.show()
else:
    print("No valid image-annotation pairs found.")

## 11. Check Data Split Folder and Summarize
We check the `data split` folder to see if there are predefined train/val/test splits.

In [ ]:
if SPLIT_DIR.exists() and any(SPLIT_DIR.iterdir()):
    split_files = [f for f in SPLIT_DIR.iterdir() if f.is_file() and not f.name.startswith('.DS_Store')]
    print("Found data split files:")
    split_summary = []
    for f in split_files:
        with open(f, 'r') as file:
            lines = file.readlines()
            print(f"- {f.name}: {len(lines)} records")
            split_summary.append({'split': f.name, 'count': len(lines)})
            
    df_splits = pd.DataFrame(split_summary)
    df_splits.to_csv(REPORT_DIR / 'data_split_summary.csv', index=False)
else:
    print("Data split directory not found or is empty.")

## 12. Save EDA Outputs
All the charts and CSVs have been successfully saved into the `reports/` folder. We can conclude our EDA here.

In [ ]:
print("EDA complete. Outputs are saved in the reports directory:")
for output_file in REPORT_DIR.iterdir():
    if not output_file.name.startswith('.'):
        print(f"- {output_file.name}")